In [0]:
%pip install azure-data-tables
%pip install azure-identity
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("azureTableName", "NBAENCUESTASRES")
dbutils.widgets.text("diasABorrar", "-30")

In [0]:
azureTableName=dbutils.widgets.get("azureTableName")
diasABorrar=int(dbutils.widgets.get("diasABorrar"))

In [0]:
from azure.data.tables import TableClient
from azure.identity import ClientSecretCredential
from datetime import datetime,timedelta,timezone
ambiente = dbutils.secrets.get(scope="secret-scope-akv", key="adb-ambiente")
scope="secret-scope-akv-"+ambiente

TENANT_ID=dbutils.secrets.get(scope=scope,key="aad-databricks-adls-app-tenantid")
CLIENT_ID=dbutils.secrets.get(scope=scope,key="aad-databricks-adls-app-clientid")
CLIENT_SECRET=dbutils.secrets.get(scope=scope,key="aad-databricks-adls-app-clientsecret")

table_url=f"https://mxapadlmd{ambiente}.table.core.windows.net"
credencial=ClientSecretCredential(tenant_id=TENANT_ID,client_id=CLIENT_ID,client_secret=CLIENT_SECRET)
table_client = TableClient(endpoint=table_url, table_name=azureTableName, credential=credencial)




In [0]:

#BORRADO
time_filtro = datetime.now() + timedelta(days=diasABorrar)
time_filtro = datetime.combine(time_filtro, datetime.max.time()).replace(tzinfo=timezone.utc)
#print(time_filtro)
#print(type(time_filtro))
entities = table_client.query_entities(query_filter="")

entidades_filtradas=[
    entity for entity in 
    entities if entity._metadata["timestamp"]<time_filtro
   ]
i=0
for entity in entidades_filtradas:
  try:
      i=i+1
      #print(f'partitionKey {entity._metadata["timestamp"]}')
      table_client.delete_entity(entity["PartitionKey"], entity["RowKey"])
      #print("borrando")
  except Exception as e:
      print(f"error {e}")
print(f"Se borraron {i} entidades")